### Layout Benchmark Evaluation — OmniDocBench JASyn

Đánh giá 3 model Layout (**`docling-layout-heron`**, **`DocLayout-YOLO`**, **`PP-DocLayoutV3`**) trên **518 trang OmniDocBench JASyn** (5 lớp nhãn tiếng Nhật).

* **`F1@0.50` / `F1@0.75`:** Trung bình điều hòa giữa *Precision* và *Recall* tại ngưỡng `IoU ≥ 0.50` / `0.75`.
* **`Precision@0.50` / `Recall@0.50`:** Tỷ lệ khung dự đoán đúng (*Precision*) và tỷ lệ khung đáp án được phát hiện (*Recall*) tại `IoU ≥ 0.50`.
* **`mAP@0.50:0.95` / `AP@0.50` / `AP@0.75`:** Độ chính xác trung bình trên tất cả các lớp qua 10 ngưỡng `IoU` từ `0.50` đến `0.95`, và riêng tại `0.50` / `0.75`.
* **`IoU` *(Intersection over Union)*:** Tỷ lệ diện tích phần giao trên phần hợp giữa khung dự đoán và khung đáp án.

In [5]:
import json
from collections import defaultdict
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display

IOU_THRESHOLDS = np.linspace(0.50, 0.95, 10)
REC_THRESHOLDS = np.linspace(0.0, 1.0, 101)
MAX_DETS = 100


def compute_iou_matrix(boxes_a: np.ndarray, boxes_b: np.ndarray) -> np.ndarray:
    if len(boxes_a) == 0 or len(boxes_b) == 0:
        return np.zeros((len(boxes_a), len(boxes_b)), dtype=np.float64)
    x1 = np.maximum(boxes_a[:, None, 0], boxes_b[None, :, 0])
    y1 = np.maximum(boxes_a[:, None, 1], boxes_b[None, :, 1])
    x2 = np.minimum(boxes_a[:, None, 2], boxes_b[None, :, 2])
    y2 = np.minimum(boxes_a[:, None, 3], boxes_b[None, :, 3])
    inter = np.maximum(0.0, x2 - x1) * np.maximum(0.0, y2 - y1)
    area_a = np.maximum(0.0, boxes_a[:, 2] - boxes_a[:, 0]) * np.maximum(0.0, boxes_a[:, 3] - boxes_a[:, 1])
    area_b = np.maximum(0.0, boxes_b[:, 2] - boxes_b[:, 0]) * np.maximum(0.0, boxes_b[:, 3] - boxes_b[:, 1])
    union = area_a[:, None] + area_b[None, :] - inter
    return np.where(union > 0.0, inter / union, 0.0)


def extract_docling_boxes(doc: dict, gt_width: float, gt_height: float, label_map: dict[str, str]) -> dict[str, list[list[float]]]:
    pages = doc.get("pages") or {}
    page_info = pages.get("1") or (next(iter(pages.values())) if pages else None)
    if not page_info or "size" not in page_info:
        return {}
    page_w = float(page_info["size"]["width"])
    page_h = float(page_info["size"]["height"])
    if page_w <= 0 or page_h <= 0:
        return {}
    scale_x, scale_y = gt_width / page_w, gt_height / page_h

    by_cat: dict[str, list[list[float]]] = defaultdict(list)
    for section in ("texts", "tables", "pictures", "key_value_items"):
        for item in doc.get(section) or ():
            raw_label = str(item.get("label", "")).strip().lower()
            mapped = label_map.get(raw_label)
            if mapped is None:
                continue
            provs = item.get("prov") or ()
            if not provs:
                continue
            bbox = provs[0].get("bbox")
            if not bbox:
                continue
            l, t, r, b = float(bbox["l"]), float(bbox["t"]), float(bbox["r"]), float(bbox["b"])
            origin = str(bbox.get("coord_origin", "BOTTOMLEFT")).upper()
            y1, y2 = (page_h - t, page_h - b) if origin == "BOTTOMLEFT" else (t, b)
            x1, x2 = min(l, r) * scale_x, max(l, r) * scale_x
            y1, y2 = min(y1, y2) * scale_y, max(y1, y2) * scale_y
            x1, x2 = min(max(x1, 0.0), gt_width), min(max(x2, 0.0), gt_width)
            y1, y2 = min(max(y1, 0.0), gt_height), min(max(y2, 0.0), gt_height)
            if x2 > x1 and y2 > y1:
                by_cat[mapped].append([x1, y1, x2, y2])
    return by_cat


def compute_ap_101(tp_flags: np.ndarray, total_gt: int) -> float:
    if total_gt == 0:
        return float("nan")
    if len(tp_flags) == 0:
        return 0.0
    tp_cum = np.cumsum(tp_flags, dtype=np.float64)
    fp_cum = np.cumsum(1.0 - tp_flags, dtype=np.float64)
    rec = tp_cum / total_gt
    prec = tp_cum / np.maximum(tp_cum + fp_cum, 1e-12)
    for i in range(len(prec) - 1, 0, -1):
        if prec[i] > prec[i - 1]:
            prec[i - 1] = prec[i]
    inds = np.searchsorted(rec, REC_THRESHOLDS, side="left")
    q = np.zeros(len(REC_THRESHOLDS), dtype=np.float64)
    valid = inds < len(prec)
    q[valid] = prec[inds[valid]]
    return float(np.mean(q))


def prf1(tp: int, fp: int, fn: int) -> tuple[float, float, float]:
    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return p, r, f1


def evaluate_subset(image_ids, gt_by_img_cat, pred_by_img_cat, cat_names):
    n_t = len(IOU_THRESHOLDS)
    idx_50, idx_75 = 0, 5
    per_cat_ap, per_cat_f1_50 = {}, {}
    total_tp_50 = total_fp_50 = total_fn_50 = 0
    total_tp_75 = total_fp_75 = total_fn_75 = 0

    for cname in cat_names:
        tp_lists = [[] for _ in range(n_t)]
        total_gt = 0
        cat_tp_50 = cat_fp_50 = cat_tp_75 = cat_fp_75 = 0

        for img_id in image_ids:
            gt_boxes = gt_by_img_cat.get((img_id, cname))
            pred_boxes = pred_by_img_cat.get((img_id, cname))
            n_gt = 0 if gt_boxes is None else len(gt_boxes)
            n_pred = 0 if pred_boxes is None else min(len(pred_boxes), MAX_DETS)
            total_gt += n_gt
            if n_pred == 0:
                continue
            if n_gt == 0:
                for t_idx in range(n_t):
                    tp_lists[t_idx].extend([0.0] * n_pred)
                cat_fp_50 += n_pred
                cat_fp_75 += n_pred
                continue

            ious = compute_iou_matrix(pred_boxes[:n_pred], gt_boxes)
            for t_idx, thr in enumerate(IOU_THRESHOLDS):
                matched_gt = np.zeros(n_gt, dtype=bool)
                tp_img = 0
                for d_idx in range(n_pred):
                    row = ious[d_idx]
                    best_g, best_iou = -1, thr - 1e-12
                    for g_idx in range(n_gt):
                        if not matched_gt[g_idx] and row[g_idx] >= best_iou:
                            best_iou = row[g_idx]
                            best_g = g_idx
                    if best_g >= 0:
                        matched_gt[best_g] = True
                        tp_lists[t_idx].append(1.0)
                        tp_img += 1
                    else:
                        tp_lists[t_idx].append(0.0)
                if t_idx == idx_50:
                    cat_tp_50 += tp_img
                    cat_fp_50 += n_pred - tp_img
                elif t_idx == idx_75:
                    cat_tp_75 += tp_img
                    cat_fp_75 += n_pred - tp_img

        aps = [compute_ap_101(np.asarray(tp_lists[t_idx], dtype=np.float64), total_gt) for t_idx in range(n_t)]
        per_cat_ap[cname] = {
            "mAP_50_95": float(np.nanmean(aps)) if not all(np.isnan(aps)) else float("nan"),
            "AP_50": aps[idx_50],
            "AP_75": aps[idx_75],
            "gt_count": total_gt,
        }
        cat_fn_50 = total_gt - cat_tp_50
        cat_fn_75 = total_gt - cat_tp_75
        _, _, per_cat_f1_50[cname] = prf1(cat_tp_50, cat_fp_50, cat_fn_50)
        total_tp_50 += cat_tp_50
        total_fp_50 += cat_fp_50
        total_fn_50 += cat_fn_50
        total_tp_75 += cat_tp_75
        total_fp_75 += cat_fp_75
        total_fn_75 += cat_fn_75

    valid_map = [v["mAP_50_95"] for v in per_cat_ap.values() if not np.isnan(v["mAP_50_95"])]
    valid_ap50 = [v["AP_50"] for v in per_cat_ap.values() if not np.isnan(v["AP_50"])]
    valid_ap75 = [v["AP_75"] for v in per_cat_ap.values() if not np.isnan(v["AP_75"])]
    p50, r50, f1_50 = prf1(total_tp_50, total_fp_50, total_fn_50)
    _, _, f1_75 = prf1(total_tp_75, total_fp_75, total_fn_75)
    return {
        "mAP_50_95": float(np.mean(valid_map)) if valid_map else 0.0,
        "AP_50": float(np.mean(valid_ap50)) if valid_ap50 else 0.0,
        "AP_75": float(np.mean(valid_ap75)) if valid_ap75 else 0.0,
        "P_50": p50,
        "R_50": r50,
        "F1_50": f1_50,
        "F1_75": f1_75,
        "per_cat_ap": per_cat_ap,
        "per_cat_f1_50": per_cat_f1_50,
    }


def fmt(val: float) -> str:
    return "-" if np.isnan(val) else f"{val * 100:.2f}%"

BASE_DIR = Path(".")
GT_PATH = BASE_DIR / "ground_truth.json"
MODELS = ["docling-layout-heron", "DocLayout-YOLO", "PP-DocLayoutV3"]
PDF_TYPE_FIELD = "layout"

# Map nhãn OmniDocBench -> Hệ nhãn chuẩn Docling (bỏ abandon & *_mask)
OMNI_GT_TO_DOCLING = {
    "text_block": "text",
    "reference": "text",
    "code_txt": "text",
    "equation_explanation": "text",
    "list_group": "text",
    "title": "section_header",
    "table": "table",
    "figure": "picture",
    "figure_caption": "caption",
    "table_caption": "caption",
    "code_txt_caption": "caption",
    "equation_caption": "caption",
    "equation_isolated": "formula",
    "equation_semantic": "formula",
    "header": "page_header",
    "footer": "page_footer",
    "page_number": "page_footer",
    "page_footnote": "footnote",
    "figure_footnote": "footnote",
    "table_footnote": "footnote",
}

# Map nhãn Docling JSON -> Cùng hệ nhãn chuẩn (gộp list_item & title để công bằng tuyệt đối)
DOCLING_LABEL_MAP = {
    "text": "text",
    "paragraph": "text",
    "code": "text",
    "reference": "text",
    "list_item": "text",
    "section_header": "section_header",
    "title": "section_header",
    "table": "table",
    "document_index": "table",
    "picture": "picture",
    "chart": "picture",
    "caption": "caption",
    "formula": "formula",
    "page_header": "page_header",
    "page_footer": "page_footer",
    "footnote": "footnote",
}

t0 = perf_counter()
gt_raw = json.loads(GT_PATH.read_text(encoding="utf-8"))

# Chỉ đánh giá các trang có tồn tại file dự đoán trong thư mục model đầu tiên
first_model_dir = next((BASE_DIR / m for m in MODELS if (BASE_DIR / m).exists()), None)
if first_model_dir is None:
    raise FileNotFoundError("Không tìm thấy thư mục model nào.")
available_stems = {
    p.stem for p in first_model_dir.rglob("*.json")
    if p.name not in ("errors.json", "current_files.json")
}

pages_by_stem = {}
doc_cat_images = defaultdict(list)
gt_lists = defaultdict(list)
present_cats = set()

for page in gt_raw:
    pinfo = page.get("page_info") or {}
    stem = Path(pinfo.get("image_path", "")).stem
    if stem not in available_stems:
        continue
    pages_by_stem[stem] = pinfo
    pattr = pinfo.get("page_attribute") or {}
    pdf_type = pattr.get(PDF_TYPE_FIELD) or "unknown"
    doc_cat_images[pdf_type].append(stem)

    for det in page.get("layout_dets") or ():
        mapped = OMNI_GT_TO_DOCLING.get(det.get("category_type", ""))
        if mapped is None:
            continue
        poly = det.get("poly")
        if poly and len(poly) >= 8:
            xs = [float(poly[i]) for i in range(0, len(poly), 2)]
            ys = [float(poly[i]) for i in range(1, len(poly), 2)]
            x1, y1, x2, y2 = min(xs), min(ys), max(xs), max(ys)
        elif det.get("bbox") and len(det["bbox"]) == 4:
            x, y, w, h = [float(v) for v in det["bbox"]]
            x1, y1, x2, y2 = x, y, x + w, y + h
        else:
            continue
        if x2 > x1 and y2 > y1:
            gt_lists[(stem, mapped)].append([x1, y1, x2, y2])
            present_cats.add(mapped)

CAT_NAMES = sorted(present_cats)
doc_categories = sorted(doc_cat_images)
all_image_ids = sorted(pages_by_stem)
gt_by_img_cat = {k: np.asarray(v, dtype=np.float64) for k, v in gt_lists.items()}

results, doc_cat_results = {}, {}
for m in MODELS:
    mdir = BASE_DIR / m
    if not mdir.exists():
        print(f"[SKIP] Không tìm thấy: {mdir}")
        continue
    stem_to_path = {
        p.stem: p for p in mdir.rglob("*.json")
        if p.name not in ("errors.json", "current_files.json")
    }
    pred_by_img_cat = {}
    missing = 0
    for stem, pinfo in pages_by_stem.items():
        pred_path = stem_to_path.get(stem)
        if pred_path is None:
            missing += 1
            continue
        doc = json.loads(pred_path.read_text(encoding="utf-8"))
        gt_w, gt_h = float(pinfo["width"]), float(pinfo["height"])
        for cname, boxes in extract_docling_boxes(doc, gt_w, gt_h, DOCLING_LABEL_MAP).items():
            if cname in present_cats:
                pred_by_img_cat[(stem, cname)] = np.asarray(boxes, dtype=np.float64)

    results[m] = evaluate_subset(all_image_ids, gt_by_img_cat, pred_by_img_cat, CAT_NAMES)
    doc_cat_results[m] = {
        dcat: evaluate_subset(doc_cat_images[dcat], gt_by_img_cat, pred_by_img_cat, CAT_NAMES)
        for dcat in doc_categories
    }
    print(f"✓ {m}: missing={missing} | mAP@50:95={fmt(results[m]['mAP_50_95'])} | F1@50={fmt(results[m]['F1_50'])}")

print(f"Hoàn tất đánh giá {len(all_image_ids)} trang ({len(CAT_NAMES)} classes, {len(doc_categories)} loại PDF) trong {perf_counter() - t0:.1f}s")

✓ docling-layout-heron: missing=0 | mAP@50:95=14.00% | F1@50=38.21%
✓ DocLayout-YOLO: missing=0 | mAP@50:95=29.84% | F1@50=44.93%
✓ PP-DocLayoutV3: missing=0 | mAP@50:95=22.77% | F1@50=36.88%
Hoàn tất đánh giá 518 trang (5 classes, 14 loại PDF) trong 3.6s


In [6]:
# Bảng 1: Tổng quan (Overall)
rows_overall = []
for m in MODELS:
    if m not in results:
        continue
    r = results[m]
    rows_overall.append({
        "Model": m,
        "mAP@0.50:0.95": fmt(r["mAP_50_95"]),
        "AP@0.50": fmt(r["AP_50"]),
        "AP@0.75": fmt(r["AP_75"]),
        "F1@0.50": fmt(r["F1_50"]),
        "Precision@0.50": fmt(r["P_50"]),
        "Recall@0.50": fmt(r["R_50"]),
        "F1@0.75": fmt(r["F1_75"]),
    })
df_overall = pd.DataFrame(rows_overall)
display(df_overall)

,Model,mAP@0.50:0.95,AP@0.50,AP@0.75,F1@0.50,Precision@0.50,Recall@0.50,F1@0.75
0,docling-layout-heron,14.00%,19.75%,15.59%,38.21%,54.21%,29.50%,32.76%
1,DocLayout-YOLO,29.84%,38.86%,32.04%,44.93%,76.28%,31.84%,40.05%
2,PP-DocLayoutV3,22.77%,30.81%,24.88%,36.88%,61.07%,26.42%,30.52%


In [7]:
# Bảng 2: Chi tiết theo từng loại Layout (F1@0.50 và COCO AP@0.50:0.95)
eval_models = [m for m in MODELS if m in results]

rows_f1 = []
rows_ap = []
for cname in CAT_NAMES:
    gt_cnt = results[eval_models[0]]["per_cat_ap"][cname]["gt_count"]
    row_f1 = {"Layout Class": cname, "GT Boxes": gt_cnt}
    row_ap = {"Layout Class": cname, "GT Boxes": gt_cnt}
    for m in eval_models:
        row_f1[m] = fmt(results[m]["per_cat_f1_50"][cname])
        row_ap[m] = fmt(results[m]["per_cat_ap"][cname]["mAP_50_95"])
    rows_f1.append(row_f1)
    rows_ap.append(row_ap)

print("=== Bảng 2a: Per-Class F1@0.50 ===")
display(pd.DataFrame(rows_f1))
print("=== Bảng 2b: Per-Class COCO AP@0.50:0.95 ===")
display(pd.DataFrame(rows_ap))

=== Bảng 2a: Per-Class F1@0.50 ===


,Layout Class,GT Boxes,docling-layout-heron,DocLayout-YOLO,PP-DocLayoutV3
0,formula,107,8.33%,29.93%,26.09%
1,picture,532,34.64%,69.93%,60.75%
2,section_header,1543,59.38%,60.12%,52.20%
3,table,406,46.70%,84.51%,78.08%
4,text,7280,32.47%,35.26%,27.76%


=== Bảng 2b: Per-Class COCO AP@0.50:0.95 ===


,Layout Class,GT Boxes,docling-layout-heron,DocLayout-YOLO,PP-DocLayoutV3
0,formula,107,0.67%,3.55%,2.20%
1,picture,532,8.89%,39.32%,31.35%
2,section_header,1543,23.85%,25.13%,17.57%
3,table,406,27.79%,66.85%,55.98%
4,text,7280,8.78%,14.33%,6.76%


In [8]:
# Bảng 3: Chi tiết theo từng loại PDF
eval_models = [m for m in MODELS if m in results]
rows_pdf = []
for dcat in doc_categories:
    row = {"PDF Type": dcat, "Pages": len(doc_cat_images[dcat])}
    for m in eval_models:
        r = doc_cat_results[m][dcat]
        row[f"{m} (mAP / F1@50)"] = f"{fmt(r['mAP_50_95'])} / {fmt(r['F1_50'])}"
    rows_pdf.append(row)

df_pdf = pd.DataFrame(rows_pdf)
display(df_pdf)

,PDF Type,Pages,docling-layout-heron (mAP / F1@50),DocLayout-YOLO (mAP / F1@50),PP-DocLayoutV3 (mAP / F1@50)
0,academic_paper,36,15.55% / 42.37%,36.65% / 53.35%,30.30% / 46.90%
1,book_vertical,55,8.11% / 30.67%,16.77% / 10.54%,16.11% / 13.55%
2,exam_paper,29,14.10% / 33.70%,33.27% / 48.26%,24.54% / 37.44%
3,handwritten_note,35,10.07% / 24.35%,26.82% / 38.77%,15.66% / 22.60%
4,lab_notebook,33,15.27% / 31.74%,32.47% / 47.86%,23.59% / 34.53%
5,magazine,37,11.79% / 37.16%,28.72% / 43.56%,16.13% / 32.12%
6,newspaper,28,18.28% / 46.44%,32.48% / 54.71%,28.10% / 45.16%
7,newspaper_vertical,21,7.45% / 28.83%,18.07% / 28.89%,10.20% / 26.63%
8,press_release,38,15.12% / 43.08%,32.17% / 49.82%,21.93% / 37.57%
9,report,42,12.36% / 32.10%,30.87% / 38.69%,26.47% / 32.65%
